# Температура по сдвигу фононного пика

1. Калибровка: положение LO i-GaAs в серии 227–374 K → модель ω(T) (Balkanski et al. 1983).
2. Применение: по положению пика в новом спектре — температура в точке лазера.

Нажмите **Run All**. Файлы `../data/raw/own/*.txt` появляются после `scripts/import_own.py`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ramancore.io import read_spectrum
from ramancore.peaks import fit_peaks
from ramancore.thermal import balkanski, fit_temperature_shift, temperature_from_shift

## 1. Калибровка по серии

In [ ]:
temps = [227, 254, 274, 294, 334, 374]
rows = []
for T in temps:
    x, y, _ = read_spectrum(f"../data/raw/own/own_gaas_vs182_{T}K.txt")
    table, _ = fit_peaks(x, y, [293], shape="voigt", margin=25)  # LO около 293 см⁻¹
    rows.append({"T, K": T, "LO, см⁻¹": table.center[0], "погрешность": table.center_err[0],
                 "FWHM": table.fwhm[0]})
series = pd.DataFrame(rows)
cal = fit_temperature_shift(series["T, K"], series["LO, см⁻¹"], series["погрешность"])
print(f"ω0 = {cal['omega0']:.2f} ± {cal['omega0_err']:.2f},  A = {cal['A']:.2f} ± {cal['A_err']:.2f}")
print(f"Средний наклон: {cal['slope']:.4f} см⁻¹/К")
series.round(2)

In [ ]:
T_grid = np.linspace(200, 400, 200)
fig = go.Figure()
fig.add_scatter(x=series["T, K"], y=series["LO, см⁻¹"], mode="markers", name="измерено",
                error_y=dict(array=series["погрешность"]))
fig.add_scatter(x=T_grid, y=balkanski(T_grid, cal["omega0"], cal["A"]), name="модель")
fig.update_layout(xaxis_title="T, K", yaxis_title="LO, см⁻¹")
fig.show()

## 2. Температура по новому спектру

Подставьте спектр того же материала (лучше того же образца), например снятый
при большей мощности лазера. Шаг прибора 1.21 см⁻¹ даёт точность центра
~0.05–0.1 см⁻¹, то есть ±2–4 K.

In [ ]:
my_file = "../data/raw/own/own_gaas_vs182_294K.txt"  # ← подставьте свой файл
x, y, _ = read_spectrum(my_file)
table, _ = fit_peaks(x, y, [293], shape="voigt", margin=25)
T, T_err, warnings = temperature_from_shift(table.center[0], cal, table.center_err[0])
print(f"LO = {table.center[0]:.2f} см⁻¹  →  T = {T:.0f} ± {T_err:.0f} K")
for w in warnings:
    print("•", w)